In [1]:
from collections import Counter
from pathlib import Path
from scipy.sparse import csr_matrix
from sklearn.model_selection import train_test_split
import numpy as np
import pandas as pd
import random
from sympy import true

import part1 # to use code from tutorial
import torch

def set_seed(seed):
    """Seeds Python, NumPy (also used by sklearn's saga solver) and PyTorch (weight init, shuffling, dropout)."""
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)

# Shared Step: Create Datasets with Three Representations

In [2]:
# splitting data into train and test sets and folds for cross-validation
LABEL_COL = "Party_orientation"
SAMPLE_FRAC = 1.0  # fraction of the data to use, 1.0 for all of it

# merge the party orientations into two classes, "Left" and "Right"
# "Centre" has very few samples, so it is discarded
orientation_to_side = {
    "Centre-left to left": "Left",
    "Centre-left": "Left",
    "Centre to centre-left": "Left",
    "Centre to centre-right": "Right",
    "Centre-right": "Right",
    "Right": "Right",
    "Right to far-right": "Right",
    "Far-right": "Right",
}

def merge_orientations(df, label_column=LABEL_COL):
    df = df[df[label_column].isin(orientation_to_side.keys())].copy()
    df[label_column] = df[label_column].map(orientation_to_side)
    return df

# get the dataframe: one dataframe with tokens, lemmas, sentences and embeddings,
# so all three representations are trained and evaluated on the same rows
df = merge_orientations(pd.read_pickle("./embeddings_sampled.pkl"))
df = df.sample(frac=SAMPLE_FRAC, random_state=42).reset_index(drop=True)

# train test split
train_df, test_df = train_test_split(df, test_size=0.1, random_state=42, stratify=df[LABEL_COL])
train_df = train_df.reset_index(drop=True)

# labels and settings
label_names = sorted(df[LABEL_COL].unique())
outcome_space = part1.clslib.NamedClasses(label_names)
# repeated cross-validation: each seed gives its own fold split and model initialisation
# (fixed in advance; more runs give the Bayesian correlated t-test more paired differences to work with)
my_seeds = [0, 1, 2]
num_folds = 5

In [3]:
# to save results
results_folder = Path("./results")
results_folder.mkdir(exist_ok=True)
# to save models
my_classifiers = {}

In [4]:
def get_vocab(texts, min_count=5, max_size=None):
    counts = Counter(item for text in texts for item in text.split())
    kept = [item for item, count in counts.most_common(max_size) if count >= min_count]
    return {item: i for i, item in enumerate(kept)}

class Encoder:
    def __len__(self) -> int:
        raise NotImplementedError

    def __call__(self, x: str) -> np.ndarray:
        raise NotImplementedError

def sparse_row(counts: dict, size: int) -> csr_matrix:
    """A 1 x size sparse row with counts[index] at each index (only the non-zero entries are stored)."""
    indices = sorted(counts)
    values = np.array([counts[i] for i in indices], dtype=np.float32)
    return csr_matrix((values, indices, [0, len(indices)]), shape=(1, size))

class TokenIndicator(Encoder):
    def __init__(self, vocab: dict):
        self.vocab = vocab

    def __len__(self) -> int:
        return len(self.vocab)

    def __call__(self, x: str) -> csr_matrix:
        # sparse row instead of np.zeros(len(vocab)), so the full dataset fits in memory
        indices = {self.vocab[item] for item in x.split() if item in self.vocab}
        return sparse_row({index: 1 for index in indices}, len(self))

# Function to create pairs of texts and labels so we can use functions from part1 (tutorial)
def make_pairs(df, column, label_names, label_column="Party_orientation"):
    label_to_id = {name: i for i, name in enumerate(label_names)}
    texts = df[column]
    # only join list columns (tokens/lemmas); str.join on a plain string column would put spaces between characters
    if isinstance(texts.iloc[0], (list, tuple, np.ndarray)):
        texts = texts.str.join(" ")
    labels = [label_to_id[name] for name in df[label_column]]
    return list(zip(texts, labels))


In [5]:
# cross-validation folds from the train rows only (the test set stays held out), using the tutorial's function
def make_cv(column):
    pairs = make_pairs(train_df, column, label_names)
    return [part1.experiment.cross_validation_split(pairs=pairs, k=num_folds, seed=seed) for seed in my_seeds]

cv_lemmas = make_cv("lemmas")        # for token indicator and bag of words
cv_sentences = make_cv("sentence")   # for the embeddings

In [6]:
# evaluation: accuracy and macro F1, per cross-validation fold and on the held-out test set
# (macro F1 averages the F1 of Left and Right, so the smaller Left class counts as much as Right)
# scores are saved to results/<experiment>/cv_scores.csv and test_scores.csv, so they survive a kernel restart
from collections import OrderedDict
from sklearn.metrics import accuracy_score, f1_score

test_lemmas = make_pairs(test_df, "lemmas", label_names)
test_sentences = make_pairs(test_df, "sentence", label_names)

def add_cv_f1(name, data_splits):
    """Adds the macro F1 of every fold's validation predictions to the runs of experiment `name`, next to "accuracies",
    and saves accuracy / macro F1 per seed and fold to results/<name>/cv_scores.csv."""
    rows = []
    for seed, run, splits in zip(my_seeds, my_classifiers[name], data_splits):
        run["f1_scores"] = OrderedDict(
            (fold, f1_score([label for _, label in splits[fold]["validation"]], preds, average="macro"))
            for fold, preds in run["predictions"].items()
        )
        for fold in run["f1_scores"]:
            rows.append({"seed": seed, "fold": fold, "accuracy": run["accuracies"][fold], "f1": run["f1_scores"][fold]})
            print(f"seed {seed} fold {fold}: accuracy {run['accuracies'][fold]:.4f} | macro F1 {run['f1_scores'][fold]:.4f}")
    pd.DataFrame(rows).to_csv(results_folder / name / "cv_scores.csv", index=False)

def load_scores(kind="cv"):
    """All saved scores of one kind ("cv" or "test"), one row per (model, seed, fold)."""
    frames = [pd.read_csv(path).assign(model=path.parent.name) for path in sorted(results_folder.glob(f"*/{kind}_scores.csv"))]
    return pd.concat(frames, ignore_index=True)[["model", "seed", "fold", "accuracy", "f1"]]

def cv_data_frame(metric="f1"):
    """One row per (model, seed, fold) with the saved cross-validation score ("accuracy" or "f1"), e.g. for the ROPE analysis."""
    return load_scores("cv").rename(columns={metric: "score"})[["model", "seed", "fold", "score"]]

def evaluate_on_test(name, test_pairs, encoder, model_option):
    """Loads every fold model of experiment `name`, predicts the test set and saves accuracy / macro F1 per model
    to results/<name>/test_scores.csv."""
    texts = [text for text, _ in test_pairs]
    gold = [label for _, label in test_pairs]
    rows = []
    for seed, run in zip(my_seeds, my_classifiers[name]):
        for fold, path in run["models"].items():
            if model_option == "logistic":
                preds = part1.loglin.predict(texts, part1.loglin.load(path), encoder)
            else:
                preds = part1.smlp.predict_probs(texts, part1.smlp.load(path), encoder, batch_size=512).argmax(axis=1)
            rows.append({"seed": seed, "fold": fold,
                         "accuracy": accuracy_score(gold, preds),
                         "f1": f1_score(gold, preds, average="macro")})
    scores = pd.DataFrame(rows)
    scores.to_csv(results_folder / name / "test_scores.csv", index=False)
    print(scores.set_index(["seed", "fold"]).round(4).to_string())
    print(f"test accuracy {scores.accuracy.mean():.4f} ± {scores.accuracy.std():.4f} | "
          f"test macro F1 {scores.f1.mean():.4f} ± {scores.f1.std():.4f}")
    return scores

In [7]:
# vocabulary from the train rows only (lemmas)
train_texts = [text for text, _ in make_pairs(train_df, "lemmas", label_names)]
vocab = get_vocab(train_texts, min_count=5, max_size=5000)

In [8]:
class BagOfWords(Encoder):
    def __init__(self, vocab: dict):
        self.vocab = vocab

    def __len__(self) -> int:
        return len(self.vocab)

    def __call__(self, x: str) -> csr_matrix:
        # repeated words are counted; sparse row so the full dataset fits in memory
        counts = Counter(self.vocab[item] for item in x.split() if item in self.vocab)
        return sparse_row(counts, len(self))

In [9]:
class Embedding(Encoder):
    dense = True  # tells part1 to use dense matrices (and lbfgs for logistic regression) instead of sparse ones

    def __init__(self, embeddings_map: pd.DataFrame):
        # build a sentence -> vector dict once, so each lookup is O(1) instead of scanning the whole dataframe
        self.lookup = {
            sentence: np.asarray(embedding, dtype=np.float32)
            for sentence, embedding in zip(embeddings_map['sentence'], embeddings_map['embeddings'])
        }
    def __len__(self) -> int:
        return 1024
    def __call__(self, x: str) -> np.ndarray:
        return self.lookup[x]

In [10]:
# encoders

token_ind_encoder = TokenIndicator(vocab)

bow_encoder = BagOfWords(vocab)

sentence_em_encoder = Embedding(df)

#print("vocabulary size:", len(token_ind_encoder))

# Experiment 1: Log-Lin

Exp 1.A: Log-Lin with Token Indicator

In [11]:
my_classifiers["logistic_indicator-lemmas"] = []

for run, seed in enumerate(my_seeds):
    set_seed(seed)
    run_folder = f"{results_folder}/logistic_indicator-lemmas/seed-{seed}"
    my_classifiers["logistic_indicator-lemmas"].append(
        part1.experiment.cross_validation_experiment(
            data_splits=cv_lemmas[run],
            outcome_space=outcome_space,
            encoder=token_ind_encoder,
            model_option="logistic",
            save_to=run_folder
        )
    )

accuracies = my_classifiers["logistic_indicator-lemmas"][0]["accuracies"]

for fold, acc in accuracies.items():
    print(f"fold {fold}: accuracy {acc:.4f}")

  0%|          | 0/5 [00:00<?, ?it/s]

/Users/chrisfeltner/Git/nlp/nlp-project/.venv/lib/python3.15/site-packages/sklearn/linear_model/_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(


/Users/chrisfeltner/Git/nlp/nlp-project/.venv/lib/python3.15/site-packages/sklearn/linear_model/_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(


/Users/chrisfeltner/Git/nlp/nlp-project/.venv/lib/python3.15/site-packages/sklearn/linear_model/_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(


/Users/chrisfeltner/Git/nlp/nlp-project/.venv/lib/python3.15/site-packages/sklearn/linear_model/_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(


/Users/chrisfeltner/Git/nlp/nlp-project/.venv/lib/python3.15/site-packages/sklearn/linear_model/_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(


... (9 outputs left out)

/Users/chrisfeltner/Git/nlp/nlp-project/.venv/lib/python3.15/site-packages/sklearn/linear_model/_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(


/Users/chrisfeltner/Git/nlp/nlp-project/.venv/lib/python3.15/site-packages/sklearn/linear_model/_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(


/Users/chrisfeltner/Git/nlp/nlp-project/.venv/lib/python3.15/site-packages/sklearn/linear_model/_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(


fold 0: accuracy 0.6925
fold 1: accuracy 0.6892
fold 2: accuracy 0.6895
fold 3: accuracy 0.6902
fold 4: accuracy 0.6901


In [12]:
add_cv_f1("logistic_indicator-lemmas", cv_lemmas)
evaluate_on_test("logistic_indicator-lemmas", test_lemmas, token_ind_encoder, "logistic");

seed 0 fold 0: accuracy 0.6925 | macro F1 0.6492
seed 0 fold 1: accuracy 0.6892 | macro F1 0.6455
seed 0 fold 2: accuracy 0.6895 | macro F1 0.6463
seed 0 fold 3: accuracy 0.6902 | macro F1 0.6467
seed 0 fold 4: accuracy 0.6901 | macro F1 0.6459
seed 1 fold 0: accuracy 0.6884 | macro F1 0.6445
seed 1 fold 1: accuracy 0.6904 | macro F1 0.6465
seed 1 fold 2: accuracy 0.6939 | macro F1 0.6503
seed 1 fold 3: accuracy 0.6914 | macro F1 0.6477
seed 1 fold 4: accuracy 0.6885 | macro F1 0.6450


seed 2 fold 0: accuracy 0.6898 | macro F1 0.6466
seed 2 fold 1: accuracy 0.6898 | macro F1 0.6467
seed 2 fold 2: accuracy 0.6901 | macro F1 0.6471
seed 2 fold 3: accuracy 0.6894 | macro F1 0.6456
seed 2 fold 4: accuracy 0.6913 | macro F1 0.6464


           accuracy      f1
seed fold                  
0    0       0.6893  0.6448
     1       0.6892  0.6446
     2       0.6890  0.6443
     3       0.6886  0.6436
     4       0.6888  0.6442
1    0       0.6903  0.6462
     1       0.6893  0.6445
     2       0.6884  0.6433
     3       0.6901  0.6458
     4       0.6881  0.6432
2    0       0.6885  0.6433
     1       0.6893  0.6445
     2       0.6879  0.6434
     3       0.6892  0.6450
     4       0.6886  0.6440
test accuracy 0.6890 ± 0.0007 | test macro F1 0.6443 ± 0.0009


Exp 1.B: Log-Lin with Bag of Words

In [13]:
my_classifiers["logistic_bow-lemmas"] = []

for run, seed in enumerate(my_seeds):
    set_seed(seed)
    run_folder = f"{results_folder}/logistic_bow-lemmas/seed-{seed}"
    my_classifiers["logistic_bow-lemmas"].append(
        part1.experiment.cross_validation_experiment(
            data_splits=cv_lemmas[run],
            outcome_space=outcome_space,
            encoder=bow_encoder,
            model_option="logistic",
            save_to=run_folder
        )
    )

accuracies = my_classifiers["logistic_bow-lemmas"][0]["accuracies"]

for fold, acc in accuracies.items():
    print(f"fold {fold}: accuracy {acc:.4f}")

  0%|          | 0/5 [00:00<?, ?it/s]

/Users/chrisfeltner/Git/nlp/nlp-project/.venv/lib/python3.15/site-packages/sklearn/linear_model/_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(


/Users/chrisfeltner/Git/nlp/nlp-project/.venv/lib/python3.15/site-packages/sklearn/linear_model/_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(


/Users/chrisfeltner/Git/nlp/nlp-project/.venv/lib/python3.15/site-packages/sklearn/linear_model/_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(


/Users/chrisfeltner/Git/nlp/nlp-project/.venv/lib/python3.15/site-packages/sklearn/linear_model/_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(


/Users/chrisfeltner/Git/nlp/nlp-project/.venv/lib/python3.15/site-packages/sklearn/linear_model/_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(


... (9 outputs left out)

/Users/chrisfeltner/Git/nlp/nlp-project/.venv/lib/python3.15/site-packages/sklearn/linear_model/_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(


/Users/chrisfeltner/Git/nlp/nlp-project/.venv/lib/python3.15/site-packages/sklearn/linear_model/_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(


/Users/chrisfeltner/Git/nlp/nlp-project/.venv/lib/python3.15/site-packages/sklearn/linear_model/_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(


fold 0: accuracy 0.6852
fold 1: accuracy 0.6874
fold 2: accuracy 0.6834
fold 3: accuracy 0.6847
fold 4: accuracy 0.6837


In [14]:
add_cv_f1("logistic_bow-lemmas", cv_lemmas)
evaluate_on_test("logistic_bow-lemmas", test_lemmas, bow_encoder, "logistic");

seed 0 fold 0: accuracy 0.6852 | macro F1 0.6305
seed 0 fold 1: accuracy 0.6874 | macro F1 0.6415
seed 0 fold 2: accuracy 0.6834 | macro F1 0.6294
seed 0 fold 3: accuracy 0.6847 | macro F1 0.6303
seed 0 fold 4: accuracy 0.6837 | macro F1 0.6285
seed 1 fold 0: accuracy 0.6825 | macro F1 0.6282
seed 1 fold 1: accuracy 0.6846 | macro F1 0.6300
seed 1 fold 2: accuracy 0.6935 | macro F1 0.6474
seed 1 fold 3: accuracy 0.6825 | macro F1 0.6271
seed 1 fold 4: accuracy 0.6836 | macro F1 0.6295


seed 2 fold 0: accuracy 0.6896 | macro F1 0.6444
seed 2 fold 1: accuracy 0.6846 | macro F1 0.6305
seed 2 fold 2: accuracy 0.6850 | macro F1 0.6315
seed 2 fold 3: accuracy 0.6827 | macro F1 0.6276
seed 2 fold 4: accuracy 0.6843 | macro F1 0.6283


           accuracy      f1
seed fold                  
0    0       0.6834  0.6279
     1       0.6895  0.6430
     2       0.6834  0.6281
     3       0.6831  0.6273
     4       0.6832  0.6282
1    0       0.6839  0.6290
     1       0.6830  0.6274
     2       0.6886  0.6415
     3       0.6830  0.6276
     4       0.6830  0.6274
2    0       0.6892  0.6421
     1       0.6835  0.6274
     2       0.6829  0.6279
     3       0.6839  0.6289
     4       0.6844  0.6292
test accuracy 0.6845 ± 0.0024 | test macro F1 0.6309 ± 0.0059


Exp 1.C: Log-Lin with Sentence Embeddings

In [15]:
my_classifiers["logistic_embeddings"] = []

for run, seed in enumerate(my_seeds):
    set_seed(seed)
    run_folder = f"{results_folder}/logistic_embeddings/seed-{seed}"
    my_classifiers["logistic_embeddings"].append(
        part1.experiment.cross_validation_experiment(
            data_splits=cv_sentences[run],
            outcome_space=outcome_space,
            encoder=sentence_em_encoder,
            model_option="logistic",
            save_to=run_folder,
            loglin_args= {
                'max_iter': 300
            }
        )
    )

accuracies = my_classifiers["logistic_embeddings"][0]["accuracies"]

for fold, acc in accuracies.items():
    print(f"fold {fold}: accuracy {acc:.4f}")

  0%|          | 0/5 [00:00<?, ?it/s]

  0%|          | 0/5 [00:00<?, ?it/s]

  0%|          | 0/5 [00:00<?, ?it/s]

fold 0: accuracy 0.6607
fold 1: accuracy 0.6551
fold 2: accuracy 0.6558
fold 3: accuracy 0.6575
fold 4: accuracy 0.6556


In [16]:
add_cv_f1("logistic_embeddings", cv_sentences)
evaluate_on_test("logistic_embeddings", test_sentences, sentence_em_encoder, "logistic");

seed 0 fold 0: accuracy 0.6607 | macro F1 0.5974
seed 0 fold 1: accuracy 0.6551 | macro F1 0.5905
seed 0 fold 2: accuracy 0.6558 | macro F1 0.5928
seed 0 fold 3: accuracy 0.6575 | macro F1 0.5931
seed 0 fold 4: accuracy 0.6556 | macro F1 0.5911
seed 1 fold 0: accuracy 0.6540 | macro F1 0.5899
seed 1 fold 1: accuracy 0.6565 | macro F1 0.5925
seed 1 fold 2: accuracy 0.6585 | macro F1 0.5938
seed 1 fold 3: accuracy 0.6585 | macro F1 0.5954
seed 1 fold 4: accuracy 0.6567 | macro F1 0.5930


seed 2 fold 0: accuracy 0.6542 | macro F1 0.5907
seed 2 fold 1: accuracy 0.6568 | macro F1 0.5929
seed 2 fold 2: accuracy 0.6576 | macro F1 0.5941
seed 2 fold 3: accuracy 0.6578 | macro F1 0.5936
seed 2 fold 4: accuracy 0.6586 | macro F1 0.5940


           accuracy      f1
seed fold                  
0    0       0.6560  0.5912
     1       0.6555  0.5907
     2       0.6562  0.5915
     3       0.6562  0.5908
     4       0.6556  0.5913
1    0       0.6555  0.5911
     1       0.6550  0.5904
     2       0.6564  0.5915
     3       0.6558  0.5908
     4       0.6567  0.5920
2    0       0.6566  0.5923
     1       0.6560  0.5908
     2       0.6553  0.5902
     3       0.6557  0.5911
     4       0.6554  0.5910
test accuracy 0.6559 ± 0.0005 | test macro F1 0.5911 ± 0.0005


# Experiment 2: Multi-Layer Perceptron
One hidden layer MLP

Exp 2.A: MLP with Token Indicator

In [17]:
my_classifiers["mlp_indicator-lemmas"] = []

for run, seed in enumerate(my_seeds):
    set_seed(seed)
    run_folder = f"{results_folder}/mlp_indicator-lemmas/seed-{seed}"
    my_classifiers["mlp_indicator-lemmas"].append(
        part1.experiment.cross_validation_experiment(
            data_splits=cv_lemmas[run],
            outcome_space=outcome_space,
            encoder=token_ind_encoder,
            model_option="mlp",
            save_to=run_folder,
            # tuned with mlp_grid_search.py / mlp_grid_check.py (see grid_search/)
            smlp_args={
                "hidden_dim": 64,
                "dropout": 0.3,
                "batch_size": 128,
                "learning_rate": 3e-4,
                "weight_decay": 1e-4,
                "num_epochs": 25,
            },
        )
    )

accuracies = my_classifiers["mlp_indicator-lemmas"][0]["accuracies"]

for fold, acc in accuracies.items():
    print(f"fold {fold}: accuracy {acc:.4f}")

  0%|          | 0/5 [00:00<?, ?it/s]

Transforming documents...


Document transformation done! Time taken: 5.215 seconds


Dataloader done! Time taken: 1.354 seconds


Training...


  0%|          | 0/25 [00:00<?, ?it/s]

... (418 outputs left out)

Epoch 22 | train loss: 0.3428 | train acc: 0.8383 | val loss: 0.6713 | val acc: 0.6935


Epoch 23 | train loss: 0.3387 | train acc: 0.8403 | val loss: 0.6807 | val acc: 0.6943


Epoch 24 | train loss: 0.3339 | train acc: 0.8431 | val loss: 0.6903 | val acc: 0.6916


Epoch 25 | train loss: 0.3300 | train acc: 0.8450 | val loss: 0.6955 | val acc: 0.6912
Training done! Time taken: 290.156 seconds
fold 0: accuracy 0.7063
fold 1: accuracy 0.7004
fold 2: accuracy 0.7037
fold 3: accuracy 0.7038
fold 4: accuracy 0.7048


In [18]:
add_cv_f1("mlp_indicator-lemmas", cv_lemmas)
evaluate_on_test("mlp_indicator-lemmas", test_lemmas, token_ind_encoder, "mlp");

seed 0 fold 0: accuracy 0.7063 | macro F1 0.6737
seed 0 fold 1: accuracy 0.7004 | macro F1 0.6692
seed 0 fold 2: accuracy 0.7037 | macro F1 0.6705
seed 0 fold 3: accuracy 0.7038 | macro F1 0.6689
seed 0 fold 4: accuracy 0.7048 | macro F1 0.6665


seed 1 fold 0: accuracy 0.7023 | macro F1 0.6700
seed 1 fold 1: accuracy 0.7038 | macro F1 0.6707
seed 1 fold 2: accuracy 0.7070 | macro F1 0.6714
seed 1 fold 3: accuracy 0.7032 | macro F1 0.6679
seed 1 fold 4: accuracy 0.7000 | macro F1 0.6586
seed 2 fold 0: accuracy 0.7038 | macro F1 0.6674
seed 2 fold 1: accuracy 0.7014 | macro F1 0.6639
seed 2 fold 2: accuracy 0.7021 | macro F1 0.6666
seed 2 fold 3: accuracy 0.7004 | macro F1 0.6641
seed 2 fold 4: accuracy 0.7044 | macro F1 0.6682


           accuracy      f1
seed fold                  
0    0       0.7005  0.6667
     1       0.7001  0.6687
     2       0.7030  0.6689
     3       0.6993  0.6640
     4       0.7031  0.6650
1    0       0.7009  0.6681
     1       0.7020  0.6684
     2       0.7014  0.6651
     3       0.7017  0.6652
     4       0.6981  0.6554
2    0       0.7001  0.6616
     1       0.6997  0.6608
     2       0.7003  0.6640
     3       0.7001  0.6630
     4       0.7017  0.6664
test accuracy 0.7008 ± 0.0014 | test macro F1 0.6648 ± 0.0036


Exp 2.B: MLP with Bag of Words

In [19]:
my_classifiers["mlp_bow-lemmas"] = []

for run, seed in enumerate(my_seeds):
    set_seed(seed)
    run_folder = f"{results_folder}/mlp_bow-lemmas/seed-{seed}"
    my_classifiers["mlp_bow-lemmas"].append(
        part1.experiment.cross_validation_experiment(
            data_splits=cv_lemmas[run],
            outcome_space=outcome_space,
            encoder=bow_encoder,
            model_option="mlp",
            save_to=run_folder,
            # tuned with mlp_grid_search.py / mlp_grid_check.py (see grid_search/)
            smlp_args={
                "hidden_dim": 64,
                "dropout": 0.3,
                "batch_size": 128,
                "learning_rate": 1e-3,
                "weight_decay": 1e-4,
                "num_epochs": 25,
            },
        )
    )

accuracies = my_classifiers["mlp_bow-lemmas"][0]["accuracies"]

for fold, acc in accuracies.items():
    print(f"fold {fold}: accuracy {acc:.4f}")

  0%|          | 0/5 [00:00<?, ?it/s]

Transforming documents...


Document transformation done! Time taken: 5.828 seconds


Dataloader done! Time taken: 1.325 seconds
Training...


  0%|          | 0/25 [00:00<?, ?it/s]

Epoch 01 | train loss: 0.5828 | train acc: 0.6812 | val loss: 0.5638 | val acc: 0.6966


... (416 outputs left out)

Epoch 22 | train loss: 0.3172 | train acc: 0.8485 | val loss: 0.8241 | val acc: 0.6832


Epoch 23 | train loss: 0.3137 | train acc: 0.8502 | val loss: 0.8425 | val acc: 0.6834


Epoch 24 | train loss: 0.3107 | train acc: 0.8509 | val loss: 0.8513 | val acc: 0.6824


Epoch 25 | train loss: 0.3064 | train acc: 0.8530 | val loss: 0.8654 | val acc: 0.6814
Training done! Time taken: 294.036 seconds
fold 0: accuracy 0.7023
fold 1: accuracy 0.6983
fold 2: accuracy 0.7023
fold 3: accuracy 0.7016
fold 4: accuracy 0.7031


In [20]:
add_cv_f1("mlp_bow-lemmas", cv_lemmas)
evaluate_on_test("mlp_bow-lemmas", test_lemmas, bow_encoder, "mlp");

seed 0 fold 0: accuracy 0.7023 | macro F1 0.6586
seed 0 fold 1: accuracy 0.6983 | macro F1 0.6585
seed 0 fold 2: accuracy 0.7023 | macro F1 0.6657
seed 0 fold 3: accuracy 0.7016 | macro F1 0.6551
seed 0 fold 4: accuracy 0.7031 | macro F1 0.6641


seed 1 fold 0: accuracy 0.7012 | macro F1 0.6604
seed 1 fold 1: accuracy 0.7033 | macro F1 0.6680
seed 1 fold 2: accuracy 0.7038 | macro F1 0.6744
seed 1 fold 3: accuracy 0.7034 | macro F1 0.6647
seed 1 fold 4: accuracy 0.7011 | macro F1 0.6643
seed 2 fold 0: accuracy 0.7005 | macro F1 0.6655
seed 2 fold 1: accuracy 0.7027 | macro F1 0.6701
seed 2 fold 2: accuracy 0.7035 | macro F1 0.6665
seed 2 fold 3: accuracy 0.7012 | macro F1 0.6695
seed 2 fold 4: accuracy 0.7040 | macro F1 0.6673


           accuracy      f1
seed fold                  
0    0       0.6990  0.6550
     1       0.7022  0.6629
     2       0.7003  0.6630
     3       0.6997  0.6527
     4       0.7004  0.6615
1    0       0.6994  0.6582
     1       0.7002  0.6645
     2       0.7002  0.6703
     3       0.7017  0.6625
     4       0.7014  0.6644
2    0       0.6993  0.6627
     1       0.7023  0.6687
     2       0.6995  0.6615
     3       0.7004  0.6685
     4       0.7011  0.6646
test accuracy 0.7005 ± 0.0010 | test macro F1 0.6627 ± 0.0048


Exp 2.C: MLP with Sentence Embeddings

In [21]:
my_classifiers["mlp_embeddings"] = []

for run, seed in enumerate(my_seeds):
    set_seed(seed)
    run_folder = f"{results_folder}/mlp_embeddings/seed-{seed}"
    my_classifiers["mlp_embeddings"].append(
        part1.experiment.cross_validation_experiment(
            data_splits=cv_sentences[run],
            outcome_space=outcome_space,
            encoder=sentence_em_encoder,
            model_option="mlp",
            save_to=run_folder,
            # tuned with mlp_grid_search.py / mlp_grid_check.py (see grid_search/): this setting was already the best
            smlp_args={
                "hidden_dim": 128,
                "dropout": 0.1,
                "batch_size": 512,
                "learning_rate": 1e-3,
                "weight_decay": 1e-4,
                "num_epochs": 25,
            },
        )
    )

accuracies = my_classifiers["mlp_embeddings"][0]["accuracies"]

for fold, acc in accuracies.items():
    print(f"fold {fold}: accuracy {acc:.4f}")

  0%|          | 0/5 [00:00<?, ?it/s]

Transforming documents...
Document transformation done! Time taken: 0.158 seconds


Dataloader done! Time taken: 0.495 seconds
Training...


  0%|          | 0/25 [00:00<?, ?it/s]

Epoch 01 | train loss: 0.6153 | train acc: 0.6511 | val loss: 0.6006 | val acc: 0.6651


Epoch 02 | train loss: 0.5946 | train acc: 0.6696 | val loss: 0.5894 | val acc: 0.6761


... (401 outputs left out)

Epoch 22 | train loss: 0.4833 | train acc: 0.7583 | val loss: 0.5860 | val acc: 0.6873


Epoch 23 | train loss: 0.4801 | train acc: 0.7617 | val loss: 0.5869 | val acc: 0.6900


Epoch 24 | train loss: 0.4773 | train acc: 0.7634 | val loss: 0.5896 | val acc: 0.6875


Epoch 25 | train loss: 0.4748 | train acc: 0.7651 | val loss: 0.5976 | val acc: 0.6888
Training done! Time taken: 38.647 seconds
fold 0: accuracy 0.6903
fold 1: accuracy 0.6880
fold 2: accuracy 0.6869
fold 3: accuracy 0.6916
fold 4: accuracy 0.6897


In [22]:
add_cv_f1("mlp_embeddings", cv_sentences)
evaluate_on_test("mlp_embeddings", test_sentences, sentence_em_encoder, "mlp");

seed 0 fold 0: accuracy 0.6903 | macro F1 0.6591
seed 0 fold 1: accuracy 0.6880 | macro F1 0.6499
seed 0 fold 2: accuracy 0.6869 | macro F1 0.6493
seed 0 fold 3: accuracy 0.6916 | macro F1 0.6541
seed 0 fold 4: accuracy 0.6897 | macro F1 0.6573
seed 1 fold 0: accuracy 0.6874 | macro F1 0.6550
seed 1 fold 1: accuracy 0.6925 | macro F1 0.6562
seed 1 fold 2: accuracy 0.6910 | macro F1 0.6549
seed 1 fold 3: accuracy 0.6888 | macro F1 0.6514
seed 1 fold 4: accuracy 0.6906 | macro F1 0.6577


seed 2 fold 0: accuracy 0.6864 | macro F1 0.6460
seed 2 fold 1: accuracy 0.6887 | macro F1 0.6498
seed 2 fold 2: accuracy 0.6905 | macro F1 0.6597
seed 2 fold 3: accuracy 0.6862 | macro F1 0.6550
seed 2 fold 4: accuracy 0.6887 | macro F1 0.6522


           accuracy      f1
seed fold                  
0    0       0.6873  0.6556
     1       0.6860  0.6470
     2       0.6878  0.6496
     3       0.6869  0.6480
     4       0.6850  0.6524
1    0       0.6890  0.6562
     1       0.6882  0.6509
     2       0.6877  0.6518
     3       0.6892  0.6503
     4       0.6887  0.6554
2    0       0.6882  0.6470
     1       0.6890  0.6495
     2       0.6888  0.6572
     3       0.6873  0.6562
     4       0.6892  0.6541
test accuracy 0.6879 ± 0.0012 | test macro F1 0.6521 ± 0.0035


In [23]:
# summary: cross-validation and test accuracy / macro F1 for every experiment (mean and std over the fold models),
# read from the score files in results/, so this also works after a kernel restart
summary = pd.concat({
    "cv": load_scores("cv").groupby("model")[["accuracy", "f1"]].agg(["mean", "std"]),
    "test": load_scores("test").groupby("model")[["accuracy", "f1"]].agg(["mean", "std"]),
}, axis=1)
majority = test_df[LABEL_COL].value_counts(normalize=True).max()
print(f"majority-class baseline: accuracy {majority:.4f}, macro F1 {majority / (1 + majority):.4f}")
summary.round(4)

majority-class baseline: accuracy 0.6210, macro F1 0.3831


cv                             test          \
                          accuracy              f1         accuracy           
                              mean     std    mean     std     mean     std   
model                                                                         
logistic_bow-lemmas         0.6851  0.0030  0.6323  0.0065   0.6845  0.0024   
logistic_embeddings         0.6569  0.0018  0.5930  0.0019   0.6559  0.0005   
logistic_indicator-lemmas   0.6903  0.0015  0.6466  0.0015   0.6890  0.0007   
mlp_bow-lemmas              0.7021  0.0015  0.6649  0.0050   0.7005  0.0010   
mlp_embeddings              0.6891  0.0019  0.6538  0.0040   0.6879  0.0012   
mlp_indicator-lemmas        0.7032  0.0021  0.6678  0.0037   0.7008  0.0014   

                                           
                               f1          
                             mean     std  
model                                      
logistic_bow-lemmas        0.6309  0.0059  
logistic_embeddings        0.5911  0.0005  
logistic_indicator-lemmas  0.6443  0.0009  
mlp_bow-lemmas             0.6627  0.0048  
mlp_embeddings             0.6521  0.0035  
mlp_indicator-lemmas       0.6648  0.0036